# Phase 04 — Naive Bayes Baseline

**Phase 04 — CS114 Final Project: Vietnamese Elderly Scam Detection**  
**Tác giả:** Bùi Thị Mỷ Cẩm (MSSV: 25730013)  
**Ngày:** 2026-09-16  

---

## Mục tiêu

Huấn luyện baseline model MultinomialNB & ComplementNB trên ma trận đặc trưng 5.015 chiều (5.000 TF-IDF + 15 aux) đã xuất sẵn từ Phase 03, đo 5 metrics (Accuracy, Precision, Recall, F1, ROC-AUC), tinh chỉnh `alpha` bằng GridSearchCV, và kiểm tra overfitting bằng Learning Curve.

> **CRITICAL INVARIANT:** KHÔNG gọi lại `train_test_split` trên toàn dataset hay `TfidfVectorizer.fit_transform` — phải nạp thẳng `X_train.pkl` / `X_test.pkl` để giữ nguyên zero-leakage protocol của Phase 03.

## 0. Setup & Imports

In [ ]:
import sys
import json
import time
from pathlib import Path

import numpy as np
import joblib

PROJECT_ROOT = Path('.').resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # kernel cwd = notebooks/ in some launchers
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import RANDOM_STATE, get_models_dir, get_reports_dir
from src.feature_engineering_config import TFIDF_MAX_FEATURES
from src.model_plots import (
    compute_metrics, plot_confusion_matrix, plot_learning_curve_chart, plot_roc_curve,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = get_reports_dir() / 'figures'
MODELS_DIR = get_models_dir()
print(f'RANDOM_STATE={RANDOM_STATE}')

## 1. Nạp Feature Matrices Từ Phase 03 (KHÔNG split lại)

In [ ]:
X_train = joblib.load(DATA_DIR / 'X_train.pkl')   # sparse CSR
X_test  = joblib.load(DATA_DIR / 'X_test.pkl')
y_train = joblib.load(DATA_DIR / 'y_train.pkl')
y_test  = joblib.load(DATA_DIR / 'y_test.pkl')

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'y_train: {len(y_train)} samples, {y_train.sum()} scam / {len(y_train)-y_train.sum()} normal')
print(f'y_test:  {len(y_test)} samples, {y_test.sum()} scam / {len(y_test)-y_test.sum()} normal')

In [ ]:
from sklearn.model_selection import train_test_split

# Validation set (15% của X_train) — chỉ dùng cho model selection, KHÔNG đụng X_test
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.15,
    random_state=RANDOM_STATE, stratify=y_train,
)
print(f'X_tr: {X_tr.shape}, X_val: {X_val.shape}')

## 2. Baseline: MultinomialNB vs ComplementNB

> ⚠️ **Lưu ý kỹ thuật:** `MultinomialNB`/`ComplementNB` giả định input là tần suất không âm (multinomial event model). 15 đặc trưng phụ trong `X_train.pkl` đã được `StandardScaler` chuẩn hóa (mean=0) nên **chứa giá trị âm** — không tương thích với 2 model này (sklearn raise `ValueError: Negative values`). Do đó với riêng NB, ta chỉ dùng khối TF-IDF không âm (5.000 cột đầu của đúng `X_train.pkl`/`X_test.pkl` đã export ở Phase 03 — KHÔNG tính toán lại gì cả, chỉ chọn subset cột tương thích thuật toán). XGBoost ở notebook kế tiếp không có ràng buộc này nên dùng đủ 5.015 chiều.

In [ ]:
from sklearn.naive_bayes import MultinomialNB, ComplementNB

# Chỉ giữ khối TF-IDF (5000 cột đầu) — tương thích non-negative cho NB
X_tr_nb, X_val_nb = X_tr[:, :TFIDF_MAX_FEATURES], X_val[:, :TFIDF_MAX_FEATURES]
X_train_nb, X_test_nb = X_train[:, :TFIDF_MAX_FEATURES], X_test[:, :TFIDF_MAX_FEATURES]
print(f'X_tr_nb: {X_tr_nb.shape}, min value: {X_tr_nb.min()}')

baselines = {
    'MultinomialNB': MultinomialNB(alpha=1.0),
    'ComplementNB': ComplementNB(alpha=1.0),
}

baseline_results = {}
for name, clf in baselines.items():
    clf.fit(X_tr_nb, y_tr)
    y_val_pred = clf.predict(X_val_nb)
    y_val_proba = clf.predict_proba(X_val_nb)[:, 1]
    metrics = compute_metrics(y_val, y_val_pred, y_val_proba)
    baseline_results[name] = metrics
    print(f'--- {name} (validation) ---')
    for k, v in metrics.items():
        print(f'  {k:10s}: {v:.4f}')

In [ ]:
from sklearn.metrics import classification_report

best_baseline_name = max(baseline_results, key=lambda n: baseline_results[n]['f1'])
print(f'Best baseline theo F1: {best_baseline_name}')
print()
print(classification_report(
    y_val, baselines[best_baseline_name].predict(X_val_nb),
    target_names=['Bình thường', 'Lừa đảo'],
))

## 3. Hyperparameter Tuning (alpha) — GridSearchCV trên X_train_nb (5-fold CV)

In [ ]:
from sklearn.model_selection import GridSearchCV

NBClass = MultinomialNB if best_baseline_name == 'MultinomialNB' else ComplementNB
param_grid = {'alpha': [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0]}

grid_search = GridSearchCV(
    NBClass(), param_grid, cv=5, scoring='f1', n_jobs=-1,
)
grid_search.fit(X_train_nb, y_train)
print(f'Best estimator: {NBClass.__name__}')
print(f'Best alpha: {grid_search.best_params_["alpha"]}')
print(f'Best CV F1: {grid_search.best_score_:.4f}')

## 4. Huấn luyện Model Cuối Cùng (best alpha, full X_train_nb) & Đánh giá trên X_test_nb

In [ ]:
nb_model = grid_search.best_estimator_  # đã tự refit trên toàn bộ X_train_nb (refit=True mặc định)

y_test_pred = nb_model.predict(X_test_nb)
y_test_proba = nb_model.predict_proba(X_test_nb)[:, 1]
test_metrics = compute_metrics(y_test, y_test_pred, y_test_proba)

print(f'--- {type(nb_model).__name__} (alpha={nb_model.alpha}) — TEST SET ---')
for k, v in test_metrics.items():
    print(f'  {k:10s}: {v:.4f}')
print()
print(classification_report(y_test, y_test_pred, target_names=['Bình thường', 'Lừa đảo']))

## 5. Confusion Matrix & ROC Curve

In [ ]:
plot_confusion_matrix(
    y_test, y_test_pred, FIGURES_DIR, 'nb_confusion_matrix.png',
    f'Confusion Matrix — {type(nb_model).__name__} (Test Set)',
)
plot_roc_curve(
    y_test, y_test_proba, FIGURES_DIR, 'nb_roc_curve.png',
    f'ROC Curve — {type(nb_model).__name__} (Test Set)',
)

## 6. Learning Curve (Overfitting Check)

In [ ]:
plot_learning_curve_chart(
    NBClass(alpha=nb_model.alpha), X_train_nb, y_train, FIGURES_DIR,
    'nb_learning_curve.png', f'Learning Curve — {type(nb_model).__name__} (alpha={nb_model.alpha})',
    cv=5, scoring='f1', random_state=RANDOM_STATE,
)

## 7. Save Model & Metrics

In [ ]:
joblib.dump(nb_model, MODELS_DIR / 'naive_bayes.pkl')

metrics_record = {
    'algorithm': type(nb_model).__name__,
    'n_features_used': X_train_nb.shape[1],
    'feature_note': 'TF-IDF only (0:TFIDF_MAX_FEATURES) — aux features excluded (negative values incompatible with NB)',
    'best_alpha': nb_model.alpha,
    'cv_f1_gridsearch': grid_search.best_score_,
    'baseline_comparison': baseline_results,
    'test_metrics': test_metrics,
    'random_state': RANDOM_STATE,
}
with open(MODELS_DIR / 'naive_bayes_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_record, f, ensure_ascii=False, indent=2)

print('Model saved: models/naive_bayes.pkl')
print('Metrics saved: models/naive_bayes_metrics.json')
print(json.dumps(metrics_record, ensure_ascii=False, indent=2))

## 8. Summary

**Phase 04 — COMPLETED** ✅

Kết quả chi tiết xem output cell 'Save Model & Metrics' phía trên (`models/naive_bayes_metrics.json`).

**Ready for:** Phase 05 (XGBoost — `04_model_xgboost.ipynb`).